# Krones Vision AI Challenge — EDA
Dataset overview, faulty / non-faulty split, out-of-ROI defects, and grouped class table.

In [ ]:
import os, json
from collections import defaultdict, Counter
import pandas as pd

ROOT = r'C:\Baksh\Masters_courses\Computer_vision\1st-krones-vision-ai-challenge'
csv = pd.read_csv(os.path.join(ROOT, 'train.csv'))
ann = json.load(open(os.path.join(ROOT, 'train_annotations.json')))
sub = pd.read_csv(os.path.join(ROOT, 'sample_submission.csv'))
cats = {c['id']: c['name'] for c in ann['categories']}
name2id = {v:k for k,v in cats.items()}
fname2id = {im['file_name']: im['id'] for im in ann['images']}
print('loaded:', len(csv), 'rows of train.csv |', len(ann['images']), 'images |', len(ann['annotations']), 'annotations')

loaded: 35342 rows of train.csv | 35342 images | 134471 annotations


## 1. Dataset length — train / test split

In [ ]:
n_train = len(os.listdir(os.path.join(ROOT, 'train_images')))
n_test  = len(os.listdir(os.path.join(ROOT, 'test_images')))
n_total = n_train + n_test
print(f'total images   : {n_total:>6}')
print(f'train images   : {n_train:>6}   ({n_train/n_total*100:5.2f}%)')
print(f'test  images   : {n_test:>6}   ({n_test /n_total*100:5.2f}%)')

total images   :  39760
train images   :  35342   (88.89%)
test  images   :   4418   (11.11%)


## 2. Faulty vs non-faulty (train)

In [ ]:
n_faulty   = int((csv.target == 1).sum())
n_reusable = int((csv.target == 0).sum())
n_csv      = len(csv)
print(f'faulty (target=1)     : {n_faulty:>6}   ({n_faulty/n_csv*100:5.2f}%)')
print(f'non-faulty (target=0) : {n_reusable:>6}   ({n_reusable/n_csv*100:5.2f}%)')

faulty (target=1)     :  20613   (58.32%)
non-faulty (target=0) :  14729   (41.68%)


## 3. Defects outside the ROI (silent loss if you train on ROI crops)

In [ ]:
ROI_ID, NOFAULT_ID = name2id['Roi'], name2id['No fault']
img_to_roi = {a['image_id']: a['bbox'] for a in ann['annotations'] if a['category_id'] == ROI_ID}

def inside(b, r):
    return (b[0] >= r[0] and b[1] >= r[1] and b[0]+b[2] <= r[0]+r[2] and b[1]+b[3] <= r[1]+r[3])

out_cnt   = Counter()
total_cnt = Counter()
for a in ann['annotations']:
    cid = a['category_id']
    if cid in (ROI_ID, NOFAULT_ID):
        continue
    total_cnt[cid] += 1
    r = img_to_roi.get(a['image_id'])
    if r is not None and not inside(a['bbox'], r):
        out_cnt[cid] += 1

rows = [(cats[c], total_cnt[c], out_cnt[c], out_cnt[c]/total_cnt[c]*100) for c in total_cnt]
df_out = (pd.DataFrame(rows, columns=['category','n_ann','n_outside_roi','pct_outside'])
            .sort_values('n_outside_roi', ascending=False)
            .reset_index(drop=True))
print(f'total defect annotations outside ROI: {sum(out_cnt.values())}\n')
df_out

total defect annotations outside ROI: 5403



,category,n_ann,n_outside_roi,pct_outside
0,No base visible,3626,3369,92.912300
1,Crown cap,2123,601,28.308997
2,Foreign object - manual cleaning,835,407,48.742515
3,Break / Crack,368,228,61.956522
4,Label,246,160,65.040650
5,Chip,717,142,19.804742
6,Contamination dark,8723,101,1.157859
7,Water drop,21751,88,0.404579
8,Mold,5981,88,1.471326
9,Yeast residue,6314,44,0.696864


## 4. Grouped class table — Defect / Harmless / Conditional

In [ ]:
img_cat = defaultdict(set)
for a in ann['annotations']:
    img_cat[a['category_id']].add(a['image_id'])
def n_imgs(name): return len(img_cat.get(name2id.get(name, -1), set()))

# Official Krones rules (from challenge PDF)
#   GOOD        -> always non-faulty
#   CONDITIONAL -> faulty only if area exceeds px threshold
#   FAULTY      -> always faulty regardless of area
GOOD = ['Embossing', 'Foam residue', 'No fault', 'Water drop']

CONDITIONAL = {
    'Air bubble':          'area > 500 px',
    'Chip':                'area > 200 px',
    'Contamination light': 'area > 180 px',
    'Glass imperfection':  'area > 100 px',
    'Scuffing':            'area > 75,000 px',
    'Scuffing heavy':      'area > 1,200 px',
}

FAULTY = ['Break / Crack', 'Circlip', 'Contamination dark', 'Crown cap',
          'Foil / Semitransparent', 'Foreign object - manual cleaning',
          'Foreign object - washing machine', 'Glass shard', 'Insect',
          'Label', 'Liquid', 'Mold', 'No base visible', 'Paint residue',
          'Straw', 'Yeast residue']

rows = []
for n in GOOD:                rows.append(('Good',        n, n_imgs(n), ''))
for n, cond in CONDITIONAL.items():
                              rows.append(('Conditional', n, n_imgs(n), cond))
for n in FAULTY:              rows.append(('Faulty',      n, n_imgs(n), ''))

df_grp = pd.DataFrame(rows, columns=['group', 'label', 'n_images', 'condition'])
group_order = {'Good': 0, 'Conditional': 1, 'Faulty': 2}
df_grp = (df_grp.assign(_o=df_grp['group'].map(group_order))
                .sort_values(['_o', 'n_images'], ascending=[True, False])
                .drop(columns='_o')
                .reset_index(drop=True))
df_grp

,group,label,n_images,condition
0,Good,Water drop,6288,
1,Good,No fault,6235,
2,Good,Foam residue,5028,
3,Good,Embossing,148,
4,Conditional,Scuffing,3080,"area > 75,000 px"
5,Conditional,Air bubble,2180,area > 500 px
6,Conditional,Contamination light,967,area > 180 px
7,Conditional,Chip,681,area > 200 px
8,Conditional,Scuffing heavy,364,"area > 1,200 px"
9,Conditional,Glass imperfection,168,area > 100 px


In [ ]:
summary = (df_grp.groupby('group')['n_images']
                  .agg(['count', 'sum'])
                  .rename(columns={'count': 'n_classes', 'sum': 'image_occurrences'})
                  .reindex(['Good', 'Conditional', 'Faulty']))
summary

,n_classes,image_occurrences
group,,
Good,4,17699
Conditional,6,7440
Faulty,16,19759
